# A Simple PyTorch Example on NAISS

The first sections will be about how to get started and run this notebook on the NAISS system of choice.

## Getting the notebook

**TODO**

## Setting up the software environment

**TODO**

## Preparing the demo

For this demo we will use the CIFAR-10 dataset and a VGG-style deep convolutional neural network. The motivation is to have an very quick and conceptually simple demo which still puts some load on the GPU.

### Dataset set-up

In [1]:
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import CIFAR10

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616),
    ),
])
dataset = CIFAR10(
    root="/mimer/NOBACKUP/Datasets/CIFAR/",
    train=True,
    download=False,
    transform=transform,
)
dataloader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=True,
    num_workers=3,
)

### Model set-up

In [2]:
from torch.nn import Sequential, Conv2d, ReLU, MaxPool2d, AdaptiveMaxPool2d, Flatten, Linear

conv_kws = {
    "kernel_size": 3,
    "stride": 1,
    "padding": "same",
}
num_classes = 10
model = Sequential(
    Conv2d(3, 64, **conv_kws), ReLU(),
    Conv2d(64, 64, **conv_kws), ReLU(),
    MaxPool2d(2),
    Conv2d(64, 128, **conv_kws), ReLU(),
    Conv2d(128, 128, **conv_kws), ReLU(),
    MaxPool2d(2),
    Conv2d(128, 256, **conv_kws), ReLU(),
    Conv2d(256, 256, **conv_kws), ReLU(),
    Conv2d(256, 256, **conv_kws), ReLU(),
    AdaptiveMaxPool2d((1, 1)),
    Flatten(start_dim=-3),
    Linear(256, 1024), ReLU(),
    Linear(1024, 1024), ReLU(),
    Linear(1024, num_classes),
)

### Training set-up
Basic set-up with

* Manual seed for [reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html)
* [Checkpointing](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html#saving-loading-model-across-devices)
* Logging to stdout

In [4]:
import torch
from torch.nn.functional import cross_entropy
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

torch.manual_seed(10037)

torch.set_float32_matmul_precision("high")
device = "cuda"
model = model.to(device)

n_epochs = 5
optim = AdamW(model.parameters(), lr=1e-3)

model.train()
for epoch in range(n_epochs):
    train_loss = 0.0
    train_acc = 0
    for batch in dataloader:
        optim.zero_grad()
        
        x = batch[0].to(device)
        y = batch[1].to(device)

        y_pred = model(x)
    
        loss = cross_entropy(y_pred, y)
        train_loss += loss.item() * x.size(0)
        train_acc += (y_pred.argmax(dim=1) == y).sum().item()
    
        loss.backward()
        optim.step()
    
    train_loss /= len(dataloader.dataset)
    train_acc /= len(dataloader.dataset)
    print(f"Epoch {epoch}, loss={train_loss}, acc={train_acc}")
    
    # Very basic checkpointing
    torch.save(
        {
            'epoch': epoch,
            'rng_state': torch.get_rng_state(),
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optim.state_dict(),
        },
        "cnn_cifar10_latest.pkl",
    )

Epoch 0, loss=2.000942763633728, acc=0.23646
Epoch 1, loss=1.5310880913925171, acc=0.42562
Epoch 2, loss=1.2704331017684936, acc=0.53042
Epoch 3, loss=1.0221214728736878, acc=0.62766
Epoch 4, loss=0.8587744745254516, acc=0.69308


**Optional excercise**: Resume training from a checkpoint <https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html>.

## Excercises
1. Launch this notebook through the Jupyter Interactive App on [Alvis OnDemand](https://alvis.c3se.chalmers.se).
    1. A single GPU and 1 or 2 hours should be enough
    2. Use the runtime from when we set-up the software environment 
2. Step through the notebook and take special notice of usage of `device` and `.to()` to make it run on the GPU
3. Monitor the GPU usage with nvtop while you rerun the notebook
    1. Open up a terminal in the Jupyter Lab instance
    2. Load the nvtop module `ml nvtop`
    3. Run `nvtop`
    4. Rerun the notebook while checking the nvtop output
4. Check the job_stats page for this job (hint: use the button next to the launch button where you launched this notebook from Alvis OnDemand)
5. Optional: Restart the training from a saved checkpoint